In [ ]:
#  SENTIMENT ANALYSIS USING LSTM (DEPLOYABLE)

In [1]:
import pandas as pd
import numpy as np
import re
import string


In [6]:
import sys
!{sys.executable} -m pip install nltk

  Using cached nltk-3.10.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.10.3-py3-none-any.whl (1.8 MB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)

   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [regex]
   ---------- ----------------------------- 1/4 [regex]
   -------------------- ------------------- 2/4 [click]
   -------------------- ------------------- 2/4 [click]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [7]:
# Text preprocessing
import nltk
from nltk.corpus import stopwords

In [8]:

# TensorFlow and Keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [9]:
# Download stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sarth\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [10]:
# 2. LOAD THE CSV DATASET
# ============================================

df = pd.read_csv("Sentiment.csv")
df 

,id,candidate,candidate_confidence,relevant_yn,relevant_yn_confidence,sentiment,sentiment_confidence,subject_matter,subject_matter_confidence,candidate_gold,...,relevant_yn_gold,retweet_count,sentiment_gold,subject_matter_gold,text,tweet_coord,tweet_created,tweet_id,tweet_location,user_timezone
0,1,No candidate mentioned,1.0000,yes,1.0000,Neutral,0.6578,None of the above,1.0000,NaN,...,NaN,5,NaN,NaN,RT @NancyLeeGrahn: How did everyone feel about...,NaN,2015-08-07 09:54:46 -0700,629697200650592256,NaN,Quito
1,2,Scott Walker,1.0000,yes,1.0000,Positive,0.6333,None of the above,1.0000,NaN,...,NaN,26,NaN,NaN,RT @ScottWalker: Didn't catch the full #GOPdeb...,NaN,2015-08-07 09:54:46 -0700,629697199560069120,NaN,NaN
2,3,No candidate mentioned,1.0000,yes,1.0000,Neutral,0.6629,None of the above,0.6629,NaN,...,NaN,27,NaN,NaN,RT @TJMShow: No mention of Tamir Rice and the ...,NaN,2015-08-07 09:54:46 -0700,629697199312482304,NaN,NaN
3,4,No candidate mentioned,1.0000,yes,1.0000,Positive,1.0000,None of the above,0.7039,NaN,...,NaN,138,NaN,NaN,RT @RobGeorge: That Carly Fiorina is trending ...,NaN,2015-08-07 09:54:45 -0700,629697197118861312,Texas,Central Time (US & Canada)
4,5,Donald Trump,1.0000,yes,1.0000,Positive,0.7045,None of the above,1.0000,NaN,...,NaN,156,NaN,NaN,RT @DanScavino: #GOPDebate w/ @realDonaldTrump...,NaN,2015-08-07 09:54:45 -0700,629697196967903232,NaN,Arizona
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13866,13867,No candidate mentioned,1.0000,yes,1.0000,Negative,0.7991,Abortion,0.6014,No candidate mentioned,...,yes,7,Negative,Abortion\nWomen's Issues (not abortion though),RT @cappy_yarbrough: Love to see men who will ...,NaN,2015-08-07 09:29:43 -0700,629690895479250944,Como,NaN
13867,13868,Mike Huckabee,0.9611,yes,1.0000,Positive,0.7302,None of the above,0.9229,Mike Huckabee,...,yes,1,NaN,NaN,RT @georgehenryw: Who thought Huckabee exceede...,NaN,2015-08-07 09:25:02 -0700,629689719056568320,USA,NaN
13868,13869,Ted Cruz,1.0000,yes,1.0000,Positive,0.8051,None of the above,0.9647,Ted Cruz,...,yes,67,Positive\nNeutral,NaN,"RT @Lrihendry: #TedCruz As President, I will a...",NaN,2015-08-07 07:19:18 -0700,629658075784282112,NaN,NaN
13869,13870,Donald Trump,1.0000,yes,1.0000,Negative,1.0000,Women's Issues (not abortion though),0.9202,Donald Trump,...,yes,149,NaN,Women's Issues (not abortion though),RT @JRehling: #GOPDebate Donald Trump says tha...,NaN,2015-08-07 09:54:04 -0700,629697023663546368,NaN,NaN


In [11]:
# Displaying first 5 rows
print(df.head())

   id               candidate  candidate_confidence relevant_yn  \
0   1  No candidate mentioned                   1.0         yes   
1   2            Scott Walker                   1.0         yes   
2   3  No candidate mentioned                   1.0         yes   
3   4  No candidate mentioned                   1.0         yes   
4   5            Donald Trump                   1.0         yes   

   relevant_yn_confidence sentiment  sentiment_confidence     subject_matter  \
0                     1.0   Neutral                0.6578  None of the above   
1                     1.0  Positive                0.6333  None of the above   
2                     1.0   Neutral                0.6629  None of the above   
3                     1.0  Positive                1.0000  None of the above   
4                     1.0  Positive                0.7045  None of the above   

   subject_matter_confidence candidate_gold  ... relevant_yn_gold  \
0                     1.0000            NaN  ..

In [14]:
# 3. Filter required columns and create a copy
# ============================================

data = df[['candidate', 'sentiment', 'text']].copy()
data

,candidate,sentiment,text
0,No candidate mentioned,Neutral,RT @NancyLeeGrahn: How did everyone feel about...
1,Scott Walker,Positive,RT @ScottWalker: Didn't catch the full #GOPdeb...
2,No candidate mentioned,Neutral,RT @TJMShow: No mention of Tamir Rice and the ...
3,No candidate mentioned,Positive,RT @RobGeorge: That Carly Fiorina is trending ...
4,Donald Trump,Positive,RT @DanScavino: #GOPDebate w/ @realDonaldTrump...
...,...,...,...
13866,No candidate mentioned,Negative,RT @cappy_yarbrough: Love to see men who will ...
13867,Mike Huckabee,Positive,RT @georgehenryw: Who thought Huckabee exceede...
13868,Ted Cruz,Positive,"RT @Lrihendry: #TedCruz As President, I will a..."
13869,Donald Trump,Negative,RT @JRehling: #GOPDebate Donald Trump says tha...


In [15]:
# Displaying dataset information
print(data.head())


                candidate sentiment  \
0  No candidate mentioned   Neutral   
1            Scott Walker  Positive   
2  No candidate mentioned   Neutral   
3  No candidate mentioned  Positive   
4            Donald Trump  Positive   

                                                text  
0  RT @NancyLeeGrahn: How did everyone feel about...  
1  RT @ScottWalker: Didn't catch the full #GOPdeb...  
2  RT @TJMShow: No mention of Tamir Rice and the ...  
3  RT @RobGeorge: That Carly Fiorina is trending ...  
4  RT @DanScavino: #GOPDebate w/ @realDonaldTrump...  


In [16]:
# 4. REMOVE NULL VALUES
# ============================================

data.dropna(inplace=True)

print("Dataset Shape:", data.shape)

Dataset Shape: (13775, 3)


In [17]:
# 5. TEXT PREPROCESSING FUNCTION
# ============================================

stop_words = set(stopwords.words('english'))

def clean_text(text):

    # Converting to lowercase
    text = text.lower()

    # Removing URLs
    text = re.sub(r"http\S+", "", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+", "", text)          # Removing mentions 
    text = re.sub(r"#", "", text)             # Removing hashtags

    # Removing punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Removing numbers
    text = re.sub(r"\d+", "", text)

    # Removing stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    # Joining words 
    text = " ".join(words)

    return text

# Apply preprocessing
data['clean_text'] = data['text'].apply(clean_text)

# Display cleaned text
print(data[['text', 'clean_text']].head())

                                                text  \
0  RT @NancyLeeGrahn: How did everyone feel about...   
1  RT @ScottWalker: Didn't catch the full #GOPdeb...   
2  RT @TJMShow: No mention of Tamir Rice and the ...   
3  RT @RobGeorge: That Carly Fiorina is trending ...   
4  RT @DanScavino: #GOPDebate w/ @realDonaldTrump...   

                                          clean_text  
0  rt everyone feel climate change question last ...  
1  rt didnt catch full gopdebate last night scott...  
2  rt mention tamir rice gopdebate held cleveland...  
3  rt carly fiorina trending hours debate men jus...  
4  rt gopdebate w delivered highest ratings histo...  


In [19]:
# Title: Converting Text Data into Numerical Sequences for NLP Model Training (TOKENIZATION)
# ============================================

MAX_WORDS = 5000

tokenizer = Tokenizer(num_words=MAX_WORDS)

tokenizer.fit_on_texts(data['clean_text'])


X = tokenizer.texts_to_sequences(data['clean_text'])

In [20]:
# 7. PADDING
# ============================================

MAX_LENGTH_SEQUENCE = 50

X = pad_sequences(X, maxlen=MAX_LENGTH_SEQUENCE)
X


array([[   0,    0,    0, ...,   14,  839,    1],
       [   0,    0,    0, ..., 1175, 1111,   72],
       [   0,    0,    0, ..., 1640,  536,  377],
       ...,
       [   0,    0,    0, ...,   38,   21,    3],
       [   0,    0,    0, ...,  896, 1334,   18],
       [   0,    0,    0, ...,   90,  108,    3]])

In [21]:
print("Padded Sequence Shape:", X.shape)

Padded Sequence Shape: (13775, 50)


In [22]:
# 8. Converting Sentiment Labels into Numeric and Categorical Format
# ============================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(data['sentiment'])

# Convert into categorical format
y = to_categorical(y)
y


array([[0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       ...,
       [0., 0., 1.],
       [1., 0., 0.],
       [0., 0., 1.]])

In [23]:
print("Encoded Labels Shape:", y.shape)

Encoded Labels Shape: (13775, 3)


In [24]:
# 9.Splitting Data into Training and Testing Sets
# ============================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42 )
    
   
print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


Training Shape: (11020, 50)
Testing Shape: (2755, 50)


In [25]:
# 10. Building an LSTM Deep Learning Model for Text Classification
# ============================================

model = Sequential()

# Embedding Layer
model.add( Embedding( input_dim=MAX_WORDS, output_dim=128))

# SpatialDropout1D Layer
model.add(SpatialDropout1D(0.2))

# LSTM Layer
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))

# Output Layer
model.add(Dense(3, activation='softmax'))

   



In [26]:
# 11. : Configuring Settings of Deep Learning Model for training
# ============================================

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)




In [27]:
# Display model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
# 12. Training the Deep Learning Model Using Training Data
# ============================================

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 16s 62ms/step - accuracy: 0.6194 - loss: 0.8812 - val_accuracy: 0.6574 - val_loss: 0.8014
Epoch 2/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.7016 - loss: 0.7041 - val_accuracy: 0.6760 - val_loss: 0.7494
Epoch 3/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - accuracy: 0.7600 - loss: 0.5856 - val_accuracy: 0.6574 - val_loss: 0.7752
Epoch 4/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - accuracy: 0.7946 - loss: 0.5044 - val_accuracy: 0.6692 - val_loss: 0.8305
Epoch 5/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.8206 - loss: 0.4480 - val_accuracy: 0.6765 - val_loss: 0.8717


In [29]:
# 13. Evaluating Model Performance on Test Data
# ============================================

loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)


87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6708 - loss: 0.8790
Test Loss: 0.8789843320846558
Test Accuracy: 0.6707804203033447


In [30]:
# Predicting Sentiment of a Text Sentence
# ============================================

sample_text = "Donald Trump performed really well in the debate"

# Clean text
sample_text = clean_text(sample_text)

# Convert into sequence
sample_sequence = tokenizer.texts_to_sequences([sample_text])

# Padding
sample_padding = pad_sequences(sample_sequence, maxlen=MAX_LENGTH_SEQUENCE)

# Prediction
prediction = model.predict(sample_padding)

# Convert prediction to label
predicted_class = np.argmax(prediction)

sentiment = label_encoder.inverse_transform([predicted_class])

print("Predicted Sentiment:", sentiment[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 615ms/step
Predicted Sentiment: Negative


In [31]:
# 15. SAVE MODEL
# ============================================

model.save("sentiment_lstm_model.keras")

print("Model Saved Successfully")

Model Saved Successfully


In [32]:
# 16. DEPLOYMENT ARTIFACT - SAVE MODEL AS model.pkl
# ==================================================
# This cell is appended only for deployment.
# All original notebook cells above remain unchanged.
#
# model.pkl stores:
# 1. LSTM model architecture
# 2. trained model weights
# 3. tokenizer vocabulary/configuration
# 4. label encoder classes
# 5. the exact stop-word set used by the notebook
# 6. sequence/model constants

import pickle

deployment_bundle = {
    "model_json": model.to_json(),
    "model_weights": model.get_weights(),
    "tokenizer": {
        "word_index": tokenizer.word_index,
        "num_words": tokenizer.num_words,
        "oov_token": getattr(tokenizer, "oov_token", None),
    },
    "label_classes": label_encoder.classes_.tolist(),
    "stop_words": sorted(list(stop_words)),
    "max_words": MAX_WORDS,
    "max_length_sequence": MAX_LENGTH_SEQUENCE,
}

with open("model.pkl", "wb") as f:
    pickle.dump(deployment_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Deployment model saved successfully as model.pkl")
print("model.pkl contains the trained LSTM weights + tokenizer + label encoder + preprocessing settings.")


Deployment model saved successfully as model.pkl
model.pkl contains the trained LSTM weights + tokenizer + label encoder + preprocessing settings.


In [33]:
# 17. VERIFY model.pkl
# ====================
# Deployment-only verification; original model/training code is untouched.

import os
import pickle

print("model.pkl exists:", os.path.exists("model.pkl"))
if os.path.exists("model.pkl"):
    print("model.pkl size:", round(os.path.getsize("model.pkl") / 1024 / 1024, 2), "MB")


model.pkl exists: True
model.pkl size: 2.96 MB
